# CitationCollapseExcelCreator

Place under gpt/prompts/

In [2]:
import json

def load_node_data(node_index):
    data = []
    with open(f"N_{node_index}_inputs.jsonl", "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [3]:
# from
node_data = [load_node_data(node_index = i) for i in range(0,12)]

# Node_data validation

In [4]:
len(node_data) == 12

True

Partition all 12 * 120 = 1440 prompts into Excel sheets of 30, for up to 48 participants. Each Excel sheet should have 900 rows, because 30 prompts * 30 papers referenced / prompt = 900 papers referenced. 

In [5]:
# 0th node, 120 prompts
len(node_data[0]) == 120

True

In [6]:
# 0th node, 0th prompt, each prompt has a citation window of 30
len(node_data[0][0]["papers_seen_id"]) == 30

True

# Stratified Assignment and Create Dataframe

In [7]:
import numpy as np

In [8]:
# (12 nodes * 120 prompts / node) / (30 prompts / person) = 48 people
# Spread out 240 prompts from two nodes evenly to 48 participants
# Stratification needs to be done by pair
def stratified_assignment(node0_data, node1_data, csvs = [list() for _ in range(48)]):

    def assign_node_data(node_data, which, csvs):
        
        for prompt_index in range(120): # 120 prompts (papers generated) per node
            if which == "first_half":
                participant_index = prompt_index % 48
            else:
                participant_index = (prompt_index + 24) % 48
            # Every prompt's citable papers must stay together
            prompt_s_citation_window_contents = []
            # loop through each prompt's citation window (each citable_paper_index)
            for citable_paper_index in range(30): # 30 papers citable per prompt
                working_paper_id = node_data[prompt_index]['id']
                type = node_data[prompt_index]['type']
                citable_paper_id = node_data[prompt_index]['papers_seen_id'][citable_paper_index]
                citable_paper_info = node_data[prompt_index]['papers_seen'][citable_paper_index]
                author = citable_paper_info['author']
                year = citable_paper_info['year']
                title = citable_paper_info['title']
                abstract = citable_paper_info['abstract']
                row = (working_paper_id, type, citable_paper_id, author, year, title, abstract, np.nan)

                prompt_s_citation_window_contents.append(row)
            
            # the n-th participant gets a prompt's data
            csvs[participant_index].extend(prompt_s_citation_window_contents)
            # if which = "first"
                # Received amounts (len(individual list inside csvs)): [3, ...(totaling 24)..., 3, 2, ...(totaling 24)..., 2]
            # if which = "second"
                # Received amounts (len(individual list inside csvs)): [5, ...(totaling 48)..., 5]

    assign_node_data(node_data = node0_data, which = "first_half", csvs = csvs)
    assign_node_data(node_data = node1_data, which = "second_half", csvs = csvs)

    return csvs

In [9]:
csvs = [list() for i in range(48)]
# loop through each node
row_amount = 0
for node_index in range(0, 12, 2): # 12 nodes
    # Assign 240 prompts from every 2 nodes, to assign evenly to 48 participants
    csvs = stratified_assignment(node_data[node_index], node_data[node_index + 1], csvs = csvs)

# Output Validation

In [10]:
# 8 Columns:
all(len(csvs[i][0]) == 8 for i in range(48))

True

In [11]:
# 900 rows per Excel sheet
all(len(csvs[i]) == 900 for i in range(48))

True

In [12]:
# (5 prompts received / pair of nodes) * 6 pairs of nodes = 30 prompts (working_paper_ids) received
valid = True
for csv_idx in range(48):
    prompts = set()
    for row_idx in range(900): 
        # 0th participant's csv, ith row, 0th entry (working_paper_id)
        prompts.add(csvs[csv_idx][row_idx][0])
    valid = valid and len(prompts) == 30
valid

True

# Create Excel

In [13]:
from pathlib import Path

In [ ]:
# for each list in csvs, create a csv under directory /human_annotation_excels
import pandas as pd
columns = ('working_paper_id', 'type', 'citable_paper_id', 'author', 'year', 'title', 'abstract', 'cite?')
for participant_index in range(48):
    csv = csvs[participant_index]
    df = pd.DataFrame(csv, columns=columns)
    filepath = Path(f"human_annotation_excels/participant_{participant_index:02d}.xlsx")
    filepath.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(filepath, index=False)